Load the dataset

In [3]:
import pandas as pd
from rags import RAG
rag = RAG()
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")

In [4]:
import time

def call_rag_with_retry(query, max_retries=3, base_delay=2):
    for attempt in range(max_retries):
        try:
            return rag.MasterRAG(query)
        except Exception as e:
            if attempt == max_retries - 1:
                # give up after final attempt, log and move on
                return {"answer": None, "error": str(e)}
            wait = base_delay * (2 ** attempt)  # exponential backoff: 2s, 4s, 8s...
            print(f"Retry {attempt+1}/{max_retries} after error: {e}. Waiting {wait}s...")
            time.sleep(wait)

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

CHECKPOINT_EVERY = 100
results = []

overall_start = time.perf_counter()

for i, row in enumerate(tqdm(meta_qa.itertuples(index=False), total=len(meta_qa))):
    start = time.perf_counter()
    art_ids, answer = call_rag_with_retry(row.question)
    elapsed = time.perf_counter() - start

    results.append({
        "query_id": row.id,
        "query": row.question,
        "answer": answer,
        "retrieved_ids": art_ids,
        "latency_sec": elapsed
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_parquet(f"output/masterRAG/outputs_partial{i+1}.parquet", index=False)

total_elapsed = time.perf_counter() - overall_start
print(f"Total time: {total_elapsed:.2f}s | Avg per query: {total_elapsed/len(meta_qa):.2f}s")

pd.DataFrame(results).to_parquet("output/masterRAG/rag_outputs.parquet", index=False)

100%|██████████| 2/2 [01:03<00:00, 31.66s/it]

Total time: 63.38s | Avg per query: 0.04s


In [6]:
outp = pd.read_parquet("output/masterRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,"Een dagvaarding is een formeel, schriftelijk o...","[15044, 14700, 4826, 3518, 7228, 4833, 13580, ...",30.057959
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: Wanneer één van de ouders allee...,"[3791, 3797, 3790, 15319, 3812, 3796, 3811, 38...",33.200534
